# 🤖 Agent IA Backend - Qwen 2.5 7B + Whisper Large v3

Serveur FastAPI pour agent IA utilisant:
- **STT**: Faster Whisper Large v3
- **LLM**: Qwen 2.5 7B Instruct
- **API**: FastAPI + ngrok
- **Frontend**: Gradio local

In [1]:
# Installation des dépendances
!pip install -q faster-whisper accelerate
!pip install -U --force-reinstall bitsandbytes # Force upgrade bitsandbytes to the latest version
!pip install -U transformers # Ensure transformers is also updated to pick up correct bitsandbytes
!pip install -q fastapi uvicorn pyngrok
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print('✅ Dépendances installées !')

  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.m

✅ Dépendances installées !


In [2]:
import os
import tempfile
import numpy as np
import asyncio
from fastapi import FastAPI, File, UploadFile, HTTPException, WebSocket
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from faster_whisper import WhisperModel
from faster_whisper.utils import download_model
import uvicorn
from pyngrok import ngrok
import torch
import json
import base64

# Configuration pour Colab
torch.cuda.empty_cache()

# Configuration des modèles
WHISPER_MODEL = "large-v3"
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
WHISPER_DIR = "/content/whisper_cache"

# Variables globales
whisper_model = None
qwen_tokenizer = None
qwen_model = None
conversation_history = []

def check_whisper_exists():
    return os.path.exists(f'{WHISPER_DIR}/large-v3') and len(os.listdir(f'{WHISPER_DIR}/large-v3')) > 3

def download_models():
    """Téléchargement intelligent des modèles"""
    if check_whisper_exists():
        print('✅ Whisper déjà présent en session, réutilisation...')
    else:
        print('⬇️ Téléchargement Whisper large-v3 (3GB)...')
        try:
            os.makedirs(WHISPER_DIR, exist_ok=True)
            download_model('large-v3', output_dir=WHISPER_DIR)
            print('✅ Whisper téléchargé')
        except Exception as e:
            print(f'❌ Erreur Whisper : {e}')

    print('🎯 Modèles prêts !')

def load_models():
    """Charger les modèles une seule fois"""
    global whisper_model, qwen_tokenizer, qwen_model

    # Clear CUDA cache before loading models to free up memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 Cache CUDA vidé.")

    if whisper_model is None:
        print("🔄 Chargement Whisper Large v3...")
        # Corrected line: Pass WHISPER_MODEL and download_root
        whisper_model = WhisperModel(
            WHISPER_MODEL,
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type="float16" if torch.cuda.is_available() else "int8",
            download_root=WHISPER_DIR
        )
        print("✅ Whisper chargé")

    if qwen_tokenizer is None or qwen_model is None:
        print("🔄 Chargement Qwen 2.5 7B (4-bit)...")
        print("⬇️ Téléchargement Qwen2.5-7B-Instruct (7GB)...")

        # Configuration 4-bit pour économiser la mémoire
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
        qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL,
            quantization_config=bnb_config if torch.cuda.is_available() else None,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True
        )
        print("✅ Qwen chargé")

def transcribe_audio(audio_file) -> str:
    """STT avec Faster Whisper Large v3"""
    try:
        if not whisper_model:
            print("Whisper model not loaded")
            return ""

        if not os.path.exists(audio_file):
            print(f"Audio file not found: {audio_file}")
            return ""

        file_size = os.path.getsize(audio_file)
        print(f"Audio file size: {file_size} bytes")

        if file_size < 100:
            print("Audio file too small")
            return ""

        segments, _ = whisper_model.transcribe(
            audio_file,
            language="fr",
            vad_filter=True,  # Re-enable VAD filter
            vad_parameters=dict(min_silence_duration_ms=500)
        )

        transcript = ""
        for segment in segments:
            transcript += segment.text.strip() + " "

        result = transcript.strip()
        print(f"Transcription result: '{result}'")
        return result

    except Exception as e:
        print(f"STT Error: {e}")
        import traceback
        traceback.print_exc()
        return ""

def generate_response(text: str) -> str:
    """LLM avec Qwen 2.5 7B"""
    global conversation_history

    if not text:
        return ""

    try:
        messages = [
            {"role": "system", "content": "Tu es un agent IA français intelligent. Réponds de manière précise et utile."}
        ]
        messages.extend(conversation_history[-6:])
        messages.append({"role": "user", "content": text})

        prompt = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = qwen_tokenizer(prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = inputs.to("cuda")

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                do_sample=True,
                pad_token_id=qwen_tokenizer.eos_token_id
            )

        response = qwen_tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        conversation_history.extend([
            {"role": "user", "content": text},
            {"role": "assistant", "content": response}
        ])

        if len(conversation_history) > 12:
            conversation_history = conversation_history[-12:]

        return response

    except Exception as e:
        print(f"Erreur LLM: {e}")
        return "Je rencontre un problème technique."

# Modèles de données
class TranscriptionResponse(BaseModel):
    transcript: str

class ChatResponse(BaseModel):
    transcript: str
    response: str

# FastAPI App
app = FastAPI(title="Agent IA Backend", version="1.0.0")

# CORS pour permettre les requêtes depuis Gradio local
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    print("WebSocket connecté")

    try:
        while True:
            data = await websocket.receive_text()
            audio_data = json.loads(data)

            if audio_data["type"] == "audio_chunk":
                # Transcription partielle rapide
                audio_bytes = base64.b64decode(audio_data["data"])

                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
                    tmp_file.write(audio_bytes)
                    tmp_path = tmp_file.name

                transcript = transcribe_audio(tmp_path)
                os.unlink(tmp_path)

                if transcript:
                    await websocket.send_text(json.dumps({
                        "type": "partial_transcript",
                        "text": transcript
                    }))
                else:
                    print("Received empty audio chunk.")

            elif audio_data["type"] == "audio_final":
                # Transcription finale + réponse IA
                audio_bytes = base64.b64decode(audio_data["data"])

                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
                    tmp_file.write(audio_bytes)
                    tmp_path = tmp_file.name

                transcript = transcribe_audio(tmp_path)
                os.unlink(tmp_path)

                if transcript:
                    await websocket.send_text(json.dumps({
                        "type": "final_transcript",
                        "text": transcript
                    }))

                    response = generate_response(transcript)

                    await websocket.send_text(json.dumps({
                        "type": "response",
                        "text": response
                    }))
                else:
                    print("Received empty final audio, no transcription or response generated.")

    except Exception as e:
        print(f"WebSocket erreur: {e}")
    finally:
        print("WebSocket déconnecté")

@app.get("/")
async def root():
    return {"message": "Agent IA Backend - Qwen 2.5 7B + Whisper Large v3"}

@app.post("/transcribe", response_model=TranscriptionResponse)
async def transcribe_endpoint(file: UploadFile = File(...)):
    """Endpoint STT"""
    try:
        # Sauvegarder le fichier temporairement
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        temp_file.write(await file.read())
        temp_file.close()

        # Transcription
        transcript = transcribe_audio(temp_file.name)

        # Nettoyer
        os.unlink(temp_file.name)

        return TranscriptionResponse(transcript=transcript)

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erreur transcription: {str(e)}")

@app.post("/chat", response_model=ChatResponse)
async def chat_endpoint(file: UploadFile = File(...)):
    """Endpoint complet: STT + LLM"""
    try:
        # Sauvegarder le fichier temporairement
        content = await file.read()
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        temp_file.write(content)
        temp_file.close()

        print(f"Received file: {file.filename}, size: {len(content)} bytes")

        # Debug: sauvegarder copie pour vérification
        debug_path = f"/content/debug_{file.filename or 'audio'}.wav"
        with open(debug_path, "wb") as f:
            f.write(content)
        print(f"Audio debug sauvé: {debug_path}")

        # STT
        transcript = transcribe_audio(temp_file.name)
        if not transcript:
            print("Empty transcription result")
            raise HTTPException(status_code=400, detail="Transcription échouée")

        # LLM
        response = generate_response(transcript)
        if not response:
            raise HTTPException(status_code=500, detail="Génération échouée")

        # Nettoyer
        os.unlink(temp_file.name)

        print(f"👂 '{transcript}' → 🤖 '{response}'")

        return ChatResponse(transcript=transcript, response=response)

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erreur chat: {str(e)}")

@app.post("/clear")
async def clear_history():
    """Effacer l'historique"""
    global conversation_history
    conversation_history.clear()
    return {"message": "Historique effacé"}

# Téléchargement des modèles
print("🚀 Initialisation du serveur Agent IA...")
download_models()

# Pré-chargement des modèles
load_models()

# ... (Tout ton code précédent pour les modèles reste identique) ...

print("✅ Modèles prêts !")
print("🌐 Préparation du serveur...")

# ==============================================================================
# CORRECTION NGROK & LANCEMENT SERVEUR
# ==============================================================================
import threading
import time

# 1. D'abord, on tue tous les anciens processus ngrok qui bloquent
ngrok.kill()

# 2. On configure le token (Ton token est inséré ici)
NGROK_TOKEN = "360Yyija3jiGh0s2Qbsw83P9yO5_4w7qqbFBa2mYDxFDWxBid"
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Fonction pour lancer FastAPI
def run_server():
    # log_level="error" pour garder la console propre
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# 4. Lancement du thread serveur
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# 5. On attend un peu que FastAPI démarre
print("⏳ Démarrage de FastAPI (attente 5s)...")
time.sleep(5)

# 6. Création du tunnel APRES le démarrage du serveur
try:
    # On force la connexion sur le port 8000
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

    print("\n" + "="*50)
    print(f"🚀 SUCCÈS ! TON BACKEND EST EN LIGNE")
    print("="*50)
    print(f"🌍 URL PUBLIQUE À COPIER : {public_url}")
    print(f"👉 Endpoint Chat : {public_url}/chat")
    print("="*50 + "\n")

except Exception as e:
    print(f"\n❌ ERREUR NGROK : {e}")
    print("Vérifie que ton token est correct et que tu n'as pas d'autres sessions ngrok actives ailleurs.")

# Boucle pour garder le script actif
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Arrêt du serveur...")
    ngrok.kill()

🚀 Initialisation du serveur Agent IA...
⬇️ Téléchargement Whisper large-v3 (3GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


✅ Whisper téléchargé
🎯 Modèles prêts !
🧹 Cache CUDA vidé.
🔄 Chargement Whisper Large v3...
✅ Whisper chargé
🔄 Chargement Qwen 2.5 7B (4-bit)...
⬇️ Téléchargement Qwen2.5-7B-Instruct (7GB)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

✅ Qwen chargé
✅ Modèles prêts !
🌐 Préparation du serveur...
⏳ Démarrage de FastAPI (attente 5s)...


INFO:     Started server process [7313]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🚀 SUCCÈS ! TON BACKEND EST EN LIGNE
🌍 URL PUBLIQUE À COPIER : https://unoffensive-mana-eustatically.ngrok-free.dev
👉 Endpoint Chat : https://unoffensive-mana-eustatically.ngrok-free.dev/chat

Received file: audio.wav, size: 176445 bytes
Audio debug sauvé: /content/debug_audio.wav.wav
Audio file size: 176445 bytes


ERROR:libav.opus:Error parsing Opus packet header.


Transcription result: 'Salut, aide-moi à calculer cette moyenne, la moyenne de 50 fois 2 plus 2.'
👂 'Salut, aide-moi à calculer cette moyenne, la moyenne de 50 fois 2 plus 2.' → 🤖 'Pour calculer la moyenne de l'expression "50 fois 2 plus 2", nous devons d'abord évaluer l'expression arithmétique.

1. Calculez "50 fois 2" : \( 50 \times 2 = 100 \).
2. Ajoutez 2 : \( 100 + 2 = 102 \).

La moyenne d'une seule valeur (ici 102) est simplement cette valeur elle-même. Donc, la moyenne de "50 fois 2 plus 2" est **102**.'
INFO:     102.38.158.36:0 - "POST /chat HTTP/1.1" 200 OK
Received file: audio.wav, size: 95595 bytes
Audio debug sauvé: /content/debug_audio.wav.wav
Audio file size: 95595 bytes
Transcription result: 'Salut, dis-moi, qui est l'actuel président du Bénin?'
👂 'Salut, dis-moi, qui est l'actuel président du Bénin?' → 🤖 'Actuellement, le président du Bénin est Patrice Talon. Il a été réélu en octobre 2021 pour un second mandat.'
INFO:     102.38.158.36:0 - "POST /chat HTTP/1.1" 200 O